# FedAcuity — EDA & Schema Exploration
**Week 1–3 Bridge** | Run after `python -m src.data.generator` completes

This notebook:
1. Validates schema constants and feature ranges
2. Runs basic EDA on the synthetic dataset (per-facility + cross-facility)
3. Visualises the non-IID distributional shift across care types
4. Confirms the mismatch label distribution hits target rates
5. Checks data splits are correctly stratified

---
> **Prerequisite**: Run `python -m src.data.generator` first to populate `data/synthetic/`

In [1]:
import os
os.chdir(r"C:\Users\iamta\OneDrive\Desktop\BITS\SEM 4\fedacuity")

import sys
sys.path.insert(0, '..')  # or set PYTHONPATH to repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from src.data.schema import (
    FEATURE_SPECS, FEATURE_NAMES, LABEL_COL,
    FACILITY_CARE_TYPES, HELD_OUT_FACILITIES,
    CLUSTER_ASSIGNMENTS, NON_IID_SPEC,
    adl_demand_score, compute_mismatch_label,
)
from src.config import cfg

SYNTHETIC_DIR = Path(cfg["paths"]["data"]["synthetic"])
SEED = cfg["project"]["seed"]

# Plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
CARE_COLORS = {"MC": "#E63946", "SNF": "#457B9D", "IL": "#2A9D8F"}

print("Imports OK")
print(f"Synthetic data dir: {SYNTHETIC_DIR}")
print(f"Dir exists: {SYNTHETIC_DIR.exists()}")

Imports OK
Synthetic data dir: data\synthetic
Dir exists: False


## 1. Schema Validation

In [2]:
# Print the full feature schema
print(f"{'Feature':<28} {'dtype':<10} {'range':<15} Description")
print("-" * 90)
for f in FEATURE_SPECS:
    print(f"{f.name:<28} {f.dtype:<10} [{f.min_val}, {f.max_val}]{'':>5} {f.description}")

print(f"\nLabel: {LABEL_COL} (binary: 1=mismatch, 0=adequate)")
print(f"Total features: {len(FEATURE_SPECS)}")

Feature                      dtype      range           Description
------------------------------------------------------------------------------------------
adl_eating                   float      [0, 6]      ADL self-performance score: eating (MDS 3.0)
adl_mobility                 float      [0, 6]      ADL self-performance score: bed mobility (MDS 3.0)
adl_toileting                float      [0, 6]      ADL self-performance score: toileting (MDS 3.0)
adl_cognition                float      [0, 6]      Cognitive function scale (MDS 3.0 BIMS proxy)
mds_adl_summary              float      [0, 28]      MDS 3.0 ADL summary score (sum of 4 ADL items × scale)
rug_category                 int        [1, 8]      RUG-IV Resource Utilization Group (1=lowest, 8=highest)
nursing_hours_rn             float      [0, 12]      Registered Nurse hours available per 24h per resident
nursing_hours_lpn            float      [0, 12]      Licensed Practical Nurse hours per 24h per resident
nursing_hours_c

In [3]:
# Validate facility assignments and cluster structure
print("Facility → Care Type Assignments")
print("-" * 40)
for fid, ct in sorted(FACILITY_CARE_TYPES.items()):
    held = " ← HELD-OUT" if fid in HELD_OUT_FACILITIES else ""
    print(f"  Facility {fid:02d}: {ct}{held}")

print("\nCluster Assignments")
print("-" * 40)
for cluster, facilities in CLUSTER_ASSIGNMENTS.items():
    print(f"  {cluster}: {facilities}")

# Sanity check: held-out facilities are NOT in any training cluster run
training_facilities = [
    fid for fid in FACILITY_CARE_TYPES
    if fid not in HELD_OUT_FACILITIES
]
print(f"\nTraining facilities ({len(training_facilities)}): {training_facilities}")
print(f"Held-out facilities: {HELD_OUT_FACILITIES}")

Facility → Care Type Assignments
----------------------------------------
  Facility 00: MC
  Facility 01: MC
  Facility 02: MC
  Facility 03: SNF
  Facility 04: SNF
  Facility 05: SNF
  Facility 06: SNF
  Facility 07: IL
  Facility 08: IL ← HELD-OUT
  Facility 09: IL ← HELD-OUT

Cluster Assignments
----------------------------------------
  MC: [0, 1, 2]
  SNF: [3, 4, 5, 6]
  IL: [7, 8, 9]

Training facilities (8): [0, 1, 2, 3, 4, 5, 6, 7]
Held-out facilities: [8, 9]


## 2. Load Synthetic Dataset

In [4]:
# Load all facilities
dfs = {}
for fid, care_type in FACILITY_CARE_TYPES.items():
    path = SYNTHETIC_DIR / f"facility_{fid:02d}_{care_type}.csv"
    if not path.exists():
        print(f"  ⚠️  Missing: {path} — run python -m src.data.generator first")
        continue
    df = pd.read_csv(path, parse_dates=["date"])
    dfs[fid] = df
    print(f"  Facility {fid:02d} [{care_type}]: {len(df)} rows, "
          f"mismatch={df[LABEL_COL].mean():.1%}")

if not dfs:
    print("\n❌ No data loaded. Run generator first.")
else:
    combined = pd.concat(dfs.values(), ignore_index=True)
    print(f"\nCombined: {len(combined)} rows, {len(dfs)} facilities")

  ⚠️  Missing: data\synthetic\facility_00_MC.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_01_MC.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_02_MC.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_03_SNF.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_04_SNF.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_05_SNF.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_06_SNF.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_07_IL.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_08_IL.csv — run python -m src.data.generator first
  ⚠️  Missing: data\synthetic\facility_09_IL.csv — run python -m src.data.generator first

❌ No data loaded. Run generator first.


## 3. Mismatch Rate by Facility

In [5]:
if dfs:
    fig, ax = plt.subplots(figsize=(10, 4))

    fids = sorted(dfs.keys())
    rates = [dfs[fid][LABEL_COL].mean() for fid in fids]
    colors = [CARE_COLORS[FACILITY_CARE_TYPES[fid]] for fid in fids]
    target_rates = {"MC": 0.40, "SNF": 0.28, "IL": 0.12}

    bars = ax.bar([f"F{fid:02d}" for fid in fids], rates, color=colors, alpha=0.85, edgecolor="white")

    # Target rate lines
    for ct, rate in target_rates.items():
        ax.axhline(rate, color=CARE_COLORS[ct], linestyle="--", alpha=0.5, linewidth=1)

    ax.set_ylabel("Mismatch Rate")
    ax.set_title("Staffing-Acuity Mismatch Rate per Facility\n(dashed = NON_IID_SPEC target)")
    ax.set_ylim(0, 0.6)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=CARE_COLORS[ct], label=ct) for ct in ["MC", "SNF", "IL"]]
    ax.legend(handles=legend_elements, loc="upper right")

    plt.tight_layout()
    plt.savefig("../results/figures/eda_mismatch_rates.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/figures/eda_mismatch_rates.png")

## 4. Non-IID Distributional Shift (Key Visualisation)

In [6]:
# The money plot: demonstrates WHY Clustered FL is needed
# Shows distributional shift across care types for the most discriminative features

if dfs:
    KEY_FEATURES = [
        "adl_cognition",
        "medication_count",
        "nursing_hours_rn",
        "mds_adl_summary",
    ]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for ax, feat in zip(axes, KEY_FEATURES):
        for care_type in ["MC", "SNF", "IL"]:
            # Pool all facilities of this care type
            ct_dfs = [dfs[fid] for fid, ct in FACILITY_CARE_TYPES.items()
                      if ct == care_type and fid in dfs]
            if not ct_dfs:
                continue
            ct_data = pd.concat(ct_dfs)
            ax.hist(
                ct_data[feat].dropna(),
                bins=30,
                alpha=0.5,
                color=CARE_COLORS[care_type],
                label=care_type,
                density=True,
            )
        ax.set_title(feat, fontsize=11)
        ax.set_ylabel("Density")
        ax.legend(fontsize=9)

    fig.suptitle(
        "Non-IID Distributional Shift Across Care Types\n"
        "(Motivates Clustered FL over standard FedAvg)",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.savefig("../results/figures/eda_noniid_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/figures/eda_noniid_distributions.png")

## 5. Feature Correlations per Care Type

In [7]:
if dfs:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    CORR_FEATURES = [
        "adl_eating", "adl_mobility", "adl_toileting", "adl_cognition",
        "medication_count", "fall_risk_score", "nursing_hours_rn",
        "nursing_hours_cna", "resident_census", LABEL_COL
    ]

    for ax, care_type in zip(axes, ["MC", "SNF", "IL"]):
        ct_dfs = [dfs[fid] for fid, ct in FACILITY_CARE_TYPES.items()
                  if ct == care_type and fid in dfs]
        if not ct_dfs:
            ax.set_visible(False)
            continue
        ct_data = pd.concat(ct_dfs)
        corr = ct_data[[f for f in CORR_FEATURES if f in ct_data.columns]].corr()
        sns.heatmap(
            corr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".1f", annot_kws={"size": 7},
            square=True, linewidths=0.5, cbar=False
        )
        ax.set_title(f"{care_type} — Correlation Matrix", fontsize=11)
        ax.tick_params(labelsize=7)

    plt.suptitle("Feature Correlation Matrices by Care Type", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("../results/figures/eda_correlation_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/figures/eda_correlation_matrices.png")

## 6. Mismatch Label Distribution

In [8]:
if dfs:
    print("Label Distribution Summary")
    print("-" * 50)
    for care_type in ["MC", "SNF", "IL"]:
        ct_dfs = [dfs[fid] for fid, ct in FACILITY_CARE_TYPES.items()
                  if ct == care_type and fid in dfs]
        if not ct_dfs:
            continue
        ct_data = pd.concat(ct_dfs)
        n = len(ct_data)
        n_mismatch = ct_data[LABEL_COL].sum()
        rate = n_mismatch / n
        target = NON_IID_SPEC[care_type]["mismatch_rate"]
        diff = abs(rate - target)
        status = "✅" if diff < 0.05 else "⚠️ "
        print(f"  {care_type}: {rate:.1%} (target {target:.0%}) {status} delta={diff:.2%}")

    # Overall
    overall = combined[LABEL_COL].mean() if 'combined' in dir() else 0
    print(f"\n  Overall: {overall:.1%} (target ~28% blended)")

## 7. Data Split Verification

In [9]:
if dfs:
    from src.data.loaders import get_facility_splits

    print("Split Verification — Checking stratification (mismatch rate preserved)")
    print("-" * 70)
    print(f"{'FID':<5} {'Care':<5} {'Train rows':<12} {'Val rows':<10} {'Test rows':<10} "
          f"{'Train mismatch':<16} {'Test mismatch'}")
    print("-" * 70)

    for fid, df in dfs.items():
        care_type = FACILITY_CARE_TYPES[fid]
        try:
            (X_train, y_train), (X_val, y_val), (X_test, y_test) = get_facility_splits(fid, df)
            print(f"  F{fid:02d}  {care_type:<5} {len(X_train):<12} {len(X_val):<10} {len(X_test):<10} "
                  f"{y_train.mean():.1%}{'':>10} {y_test.mean():.1%}")
        except Exception as e:
            print(f"  F{fid:02d}  ERROR: {e}")

    print("\n✅ Stratification check: train/test mismatch rates should be similar")

## 8. ADL Demand Score Distribution

In [10]:
# Visualise the intermediate ADL demand score — the key component of the mismatch label formula
if dfs:
    fig, ax = plt.subplots(figsize=(10, 4))

    for care_type in ["MC", "SNF", "IL"]:
        ct_dfs = [dfs[fid] for fid, ct in FACILITY_CARE_TYPES.items()
                  if ct == care_type and fid in dfs]
        if not ct_dfs:
            continue
        ct_data = pd.concat(ct_dfs)
        demand = adl_demand_score(
            ct_data["adl_eating"].values,
            ct_data["adl_mobility"].values,
            ct_data["adl_toileting"].values,
            ct_data["adl_cognition"].values,
        )
        ax.hist(demand, bins=40, alpha=0.5, color=CARE_COLORS[care_type],
                label=f"{care_type} (mean={demand.mean():.2f})", density=True)

    ax.set_xlabel("ADL Demand Score (0–1 normalised)")
    ax.set_ylabel("Density")
    ax.set_title("ADL Demand Score Distribution by Care Type\n"
                 "(Input to mismatch label formula)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../results/figures/eda_adl_demand_score.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Summary Statistics Table

In [11]:
if dfs:
    # Per care type summary stats — this becomes Table I in the paper
    summary_rows = []
    for care_type in ["MC", "SNF", "IL"]:
        ct_dfs = [dfs[fid] for fid, ct in FACILITY_CARE_TYPES.items()
                  if ct == care_type and fid in dfs]
        if not ct_dfs:
            continue
        ct_data = pd.concat(ct_dfs)
        row = {
            "Care Type": care_type,
            "N Facilities": len(ct_dfs),
            "N Records": len(ct_data),
            "Mismatch Rate": f"{ct_data[LABEL_COL].mean():.1%}",
            "ADL Cognition (mean)": f"{ct_data['adl_cognition'].mean():.2f}",
            "Medication Count (mean)": f"{ct_data['medication_count'].mean():.1f}",
            "Nursing Hours RN (mean)": f"{ct_data['nursing_hours_rn'].mean():.2f}",
            "Resident Census (mean)": f"{ct_data['resident_census'].mean():.0f}",
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)  # noqa: F821

    # Save for paper
    summary_df.to_csv("../results/tables/table1_dataset_summary.csv", index=False)
    print("Saved: results/tables/table1_dataset_summary.csv")

---
## EDA Checklist

- [ ] Schema validates — all features within expected ranges
- [ ] 10 facility CSVs exist in `data/synthetic/`
- [ ] Mismatch rates: MC ~40%, SNF ~28%, IL ~12% (±5% acceptable)
- [ ] Non-IID shift is visually clear in distribution plots
- [ ] Correlation matrices differ across care types
- [ ] Train/test splits preserve stratification
- [ ] `table1_dataset_summary.csv` saved to `results/tables/`

**When all boxes are checked → Week 4 / Phase 2 data engineering is complete.**

*Last updated: Session 3*